In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from typing import List, Dict
import os
import sys
from urllib.parse import urljoin
from pprint import pprint
import re

BASE_PATH = "../../data/RAG"

JOB_BASE_URL = "https://maplestory.nexon.com/Guide/N23Job"

job_categories = {
    "전사": 1,
    "마법사": 2,
    "궁수": 3,
    "도적": 4,
    "해적": 5,
}

In [ ]:
def get_job_urls(category: str, category_id: int):
    url = f"{JOB_BASE_URL}/{category_id}"

    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) " "Chrome/120.0.0.0")
    }

    response = requests.get(url, headers=headers, timeout=20)

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    jobs = []

    links = soup.select("a[href*='/Guide/N23Job/View/']")

    for link in links:
        href = link.get("href")
        name = link.get_text(strip=True)

        jobs.append({"category": category, "name": name, "url": urljoin(url, href)})

    return jobs

In [6]:
warrior_jobs = get_job_urls("전사", 1)

print("전사 직업 수:", len(warrior_jobs))

for job in warrior_jobs:
    print(job)

전사 직업 수: 13
{'category': '전사', 'name': '히어로', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/1'}
{'category': '전사', 'name': '팔라딘', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/2'}
{'category': '전사', 'name': '다크나이트', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/3'}
{'category': '전사', 'name': '소울마스터', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/4'}
{'category': '전사', 'name': '미하일', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/5'}
{'category': '전사', 'name': '블래스터', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/6'}
{'category': '전사', 'name': '데몬 슬레이어', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/7'}
{'category': '전사', 'name': '데몬 어벤져', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/8'}
{'category': '전사', 'name': '아란', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/9'}
{'category': '전사', 'name': '카이저', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/10'}
{'category': '전사', 'name': '아델', 'url': 'https

In [ ]:
all_job_urls = []

for category, category_id in job_categories.items():

    jobs = get_job_urls(category, category_id)

    all_job_urls.extend(jobs)

    print(f"{category}: {len(jobs)}개 직업 발견")

print("전체 직업 링크 수:", len(all_job_urls))

전사: 13개 직업 발견
마법사: 11개 직업 발견
궁수: 7개 직업 발견
도적: 9개 직업 발견
해적: 9개 직업 발견
전체 직업 링크 수: 49


In [ ]:
def get_job_page(url: str) -> str:

    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) " "Chrome/120.0.0.0")
    }

    try:
        response = requests.get(url, headers=headers, timeout=20)

        response.raise_for_status()

        return response.text

    except requests.exceptions.RequestException as e:
        print(f"직업 페이지 요청 중 오류: {e}")
        return ""

In [10]:
def parse_job_info(html: str, url: str, category: str) -> Dict:
    soup = BeautifulSoup(html, "html.parser")

    # 직업 상세 영역
    job_area = soup.select_one(".char_view")

    if job_area is None:
        print("직업 정보를 찾지 못했습니다.")
        return {}

    # 직업명
    name_tag = job_area.select_one("h2")
    name = name_tag.get_text(strip=True) if name_tag else ""

    # 직업 한 줄 소개
    subtitle_tag = job_area.select_one("h3")
    subtitle = subtitle_tag.get_text(strip=True) if subtitle_tag else ""

    # 상세 설명
    description_tag = job_area.select_one(".char_v_txt")
    description = (
        description_tag.get_text(
            separator=" ",
            strip=True
        )
        if description_tag else ""
    )

    # 스탯 정보
    stats = {}

    stat_items = job_area.select("ul.stat dl")

    for item in stat_items:
        dt = item.select_one("dt")
        dd = item.select_one("dd")

        if dt and dd:
            key = dt.get_text(strip=True)
            value = dd.get_text(
                separator=" ",
                strip=True
            )

            stats[key] = value

    # URL에서 직업 ID
    job_id = url.rstrip("/").split("/")[-1]

    job = {
        "job_id": job_id,
        "category": category,
        "name": name,
        "subtitle": subtitle,
        "description": description,
        "max_level": stats.get("최고레벨", ""),
        "main_stat": stats.get("주요스탯", ""),
        "weapon": stats.get("사용무기", ""),
        "url": url
    }

    return job

In [13]:
all_jobs = []

for item in all_job_urls:

    html = get_job_page(item["url"])

    if not html:
        continue

    job = parse_job_info(html, item["url"], item["category"])

    if job:
        all_jobs.append(job)

        print(f"{job['category']} - " f"{job['name']} 수집 완료")

print("전체 직업 수:", len(all_jobs))

전사 - 히어로 수집 완료
전사 - 팔라딘 수집 완료
전사 - 다크나이트 수집 완료
전사 - 소울마스터 수집 완료
전사 - 미하일 수집 완료
전사 - 블래스터 수집 완료
전사 - 데몬 슬레이어 수집 완료
전사 - 데몬 어벤져 수집 완료
전사 - 아란 수집 완료
전사 - 카이저 수집 완료
전사 - 아델 수집 완료
전사 - 렌 수집 완료
전사 - 제로 수집 완료
마법사 - 아크메이지(불,독) 수집 완료
마법사 - 아크메이지(썬,콜) 수집 완료
마법사 - 비숍 수집 완료
마법사 - 플레임위자드 수집 완료
마법사 - 배틀메이지 수집 완료
마법사 - 에반 수집 완료
마법사 - 루미너스 수집 완료
마법사 - 일리움 수집 완료
마법사 - 라라 수집 완료
마법사 - 키네시스 수집 완료
마법사 - 레테 수집 완료
궁수 - 보우마스터 수집 완료
궁수 - 신궁 수집 완료
궁수 - 패스파인더 수집 완료
궁수 - 윈드브레이커 수집 완료
궁수 - 와일드헌터 수집 완료
궁수 - 메르세데스 수집 완료
궁수 - 카인 수집 완료
도적 - 나이트로드 수집 완료
도적 - 섀도어 수집 완료
도적 - 듀얼블레이드 수집 완료
도적 - 나이트워커 수집 완료
도적 - 제논 수집 완료
도적 - 팬텀 수집 완료
도적 - 카데나 수집 완료
도적 - 칼리 수집 완료
도적 - 호영 수집 완료
해적 - 바이퍼 수집 완료
해적 - 캡틴 수집 완료
해적 - 캐논슈터 수집 완료
해적 - 스트라이커 수집 완료
해적 - 메카닉 수집 완료
해적 - 제논 수집 완료
해적 - 은월 수집 완료
해적 - 엔젤릭버스터 수집 완료
해적 - 아크 수집 완료
전체 직업 수: 49


In [14]:
def save_jobs_to_json(guides, BASE_PATH, file_name="maple_jobs.json"):

    file_path = os.path.join(BASE_PATH, file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(guides, f, ensure_ascii=False, indent=2)

    print("JSON 저장 완료")
    return file_path


save_jobs_to_json(all_jobs, BASE_PATH)

JSON 저장 완료


'../../data/RAG\\maple_jobs.json'